In [1]:
import sys
sys.path.insert(1, '../')
import helpers
from helpers.crossattention import CrossAttentionSiamese

import os
import torch
from torchvision import transforms
from pathlib import Path

# Import analysis functions (save the artifact as retroactive_analysis.py)
import debug_retroactive_analysis as ra

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
models_path = "logs/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/"
CHECKPOINT_DIR = os.path.join(models_path, "checkpoints")
train_set = models_path.split("/")[2]
MODEL_TYPE = models_path.split("/")[3]
image_process = models_path.split("/")[4]
run_name = models_path.split("/")[5]
MODEL_NAME = run_name.split("_")[0]
if MODEL_TYPE == "cross_attention":
    weights = run_name.split("_")[0]
    NUM_ATTN_LAYERS = int(run_name.split("_")[2].split("L")[0][-1:])
    NUM_HEADS = int(run_name.split("_")[2].split("L")[1][0])
    use_symm = True if run_name.split("_")[4] == "sym" else False
    run_id = models_path.strip("/")[-1:]
else:
    raise NotImplementedError(f"Model type \"{MODEL_NAME}\" unsupported! Model Type must be: cross_attention ")
RESIZE_DIM = 224
OUTPUT_DIR = os.path.join(models_path.replace("logs","plots"), "analysis_outputs")
FREEZE_BACKBONE = True
BATCH_SIZE = 64
WORKERS = 1
DATASET_BASE_PATH = f"../split_node21_sets/{image_process}"

In [3]:
ANALYZE_BEST_ONLY = False
if MODEL_NAME == "single":
    transform = transforms.Compose([
        transforms.Grayscale(1),
        transforms.Resize((RESIZE_DIM, RESIZE_DIM)),
        transforms.ToTensor(),
    ])
else:
    transform = transforms.Compose([
        transforms.Resize((RESIZE_DIM, RESIZE_DIM)),
        transforms.ToTensor(),
    ])
datasets_config = {
    'train-chestxray14': f"{DATASET_BASE_PATH}/chestxray14/train",
    'test-chestxray14': f"{DATASET_BASE_PATH}/chestxray14/test",
    'test-jsrt': f"{DATASET_BASE_PATH}/jsrt/test",
    'test-padchest': f"{DATASET_BASE_PATH}/padchest/test",
}
dataloaders = {}
for name, path in datasets_config.items():
    print(f"\nLoading {name}...")
    dataloaders[name] = helpers.dataloading.load_image_pair_dataset(
        dataset_path=path,
        batch_size=BATCH_SIZE,
        crop_size=RESIZE_DIM,
        symmetrical_transforms=False,
        class_to_idx={'nodule': 1, 'normal': 0},
        transform=transform,
        cache_in_ram=False,
        single=(MODEL_NAME == "single"),
        num_workers=WORKERS
    )

print("Building model...")
backbone = helpers.models.load_truncated_model(MODEL_NAME)

if MODEL_TYPE == "siamese":
    model = helpers.models.SiameseNetwork(
        backbone,
        embedding_dim=128,
        freeze_backbone=FREEZE_BACKBONE
    ).to(device)
elif MODEL_TYPE == "cross_attention":
    model = CrossAttentionSiamese(
        backbone,
        embedding_dim=128,
        num_attn_layers=NUM_ATTN_LAYERS,
        num_heads=NUM_HEADS,
        freeze_backbone=FREEZE_BACKBONE
    ).to(device)

print(f"Model: {MODEL_TYPE}")


Loading train-chestxray14...


Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 57.36it/s]


Total pairs: 1239
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 420
  normal (idx=0): 819

Loading test-chestxray14...


Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 122.58it/s]


Total pairs: 531
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 180
  normal (idx=0): 351

Loading test-jsrt...


...

Total pairs: 72
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 44
  normal (idx=0): 28

Loading test-padchest...


Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 140.40it/s]

Total pairs: 504
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 94
  normal (idx=0): 410
Building model...



/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model: cross_attention


In [4]:
checkpoint_files = sorted(Path(CHECKPOINT_DIR).glob("checkpoint_epoch_*.pth"))
print(f"\nFound {len(checkpoint_files)} checkpoint(s) to analyze")


Found 30 checkpoint(s) to analyze


In [5]:
checkpoint_path = checkpoint_files[-1]

In [6]:
# %% Debug Cell - Check Model Structure
print("="*80)
print("MODEL STRUCTURE CHECK")
print("="*80)

# Check if model has cross-attention
print(f"\nHas 'cross_attn_layers': {hasattr(model, 'cross_attn_layers')}")
if hasattr(model, 'cross_attn_layers'):
    print(f"Number of attention layers: {len(model.cross_attn_layers)}")
    print(f"Attention layers type: {type(model.cross_attn_layers)}")

print(f"Has 'get_attention_maps': {hasattr(model, 'get_attention_maps')}")

# Check what's in the state dict
checkpoint = torch.load(str(checkpoint_path), map_location=device)
state_dict_keys = list(checkpoint['model_state_dict'].keys())
print(f"\nState dict has {len(state_dict_keys)} keys")
print("\nKeys related to attention:")
attn_keys = [k for k in state_dict_keys if 'attn' in k.lower()]
if attn_keys:
    for key in attn_keys[:10]:  # Show first 10
        print(f"  {key}")
else:
    print("  No attention-related keys found!")

# Try loading and checking
model.load_state_dict(checkpoint['model_state_dict'])
print(f"\nAfter loading checkpoint:")
print(f"Has 'cross_attn_layers': {hasattr(model, 'cross_attn_layers')}")
if hasattr(model, 'cross_attn_layers'):
    print(f"Number of attention layers: {len(model.cross_attn_layers)}")

# Test forward pass
print("\n" + "="*80)
print("FORWARD PASS TEST")
print("="*80)
model.eval()
with torch.no_grad():
    for img1, img2, labels, _, _ in dataloaders['test-chestxray14']:
        img1, img2 = img1[:2].to(device), img2[:2].to(device)
        
        # Regular forward
        emb1, emb2 = model(img1, img2)
        print(f"Embeddings shape: {emb1.shape}, {emb2.shape}")
        print(f"Embeddings range: [{emb1.min():.4f}, {emb1.max():.4f}]")
        
        # Try attention maps
        if hasattr(model, 'get_attention_maps'):
            try:
                attn = model.get_attention_maps(img1, img2, layer_idx=0)
                print(f"\nAttention maps shape: {attn.shape}")
                print(f"Attention range: [{attn.min():.4f}, {attn.max():.4f}]")
                print(f"Attention contains NaN: {torch.isnan(attn).any()}")
                print(f"Attention contains Inf: {torch.isinf(attn).any()}")
            except Exception as e:
                print(f"\nERROR getting attention maps: {e}")
                import traceback
                traceback.print_exc()
        
        break

MODEL STRUCTURE CHECK

Has 'cross_attn_layers': True
Number of attention layers: 1
Attention layers type: <class 'torch.nn.modules.container.ModuleList'>
Has 'get_attention_maps': True

State dict has 338 keys

Keys related to attention:
  cross_attn_layers.0.query.weight
  cross_attn_layers.0.query.bias
  cross_attn_layers.0.key.weight
  cross_attn_layers.0.key.bias
  cross_attn_layers.0.value.weight
  cross_attn_layers.0.value.bias
  cross_attn_layers.0.out_proj.weight
  cross_attn_layers.0.out_proj.bias
  cross_attn_layers.0.norm1.weight
  cross_attn_layers.0.norm1.bias

After loading checkpoint:
Has 'cross_attn_layers': True
Number of attention layers: 1

FORWARD PASS TEST
Embeddings shape: torch.Size([2, 128]), torch.Size([2, 128])
Embeddings range: [-0.2368, 0.2788]

Attention maps shape: torch.Size([2, 1, 1, 1, 1])
Attention range: [1.0000, 1.0000]
Attention contains NaN: False
Attention contains Inf: False


Embeddings are 1D - pooled! Supposed to be unpooled for cross attention, so it can compare regions not the full embedding.

*Updating `models.py` -> `load_truncated_model()` to accept a flag that by default is True, so all prior runs are using 1D embedding, and by passing in False, will output unpooled embedding*

In [7]:
# Test script
import sys
sys.path.insert(1, '../')
import torch
import helpers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Test both modes
print("Testing single_embedding=True (for Siamese):")
backbone_1x1 = helpers.models.load_truncated_model("rad", single_embedding=True)
test_input = torch.randn(2, 3, 224, 224).to(device)
output_1x1 = backbone_1x1(test_input)
print(f"  Output shape: {output_1x1.shape}")
print(f"  Expected: torch.Size([2, 2048, 1, 1])")

print("\nTesting single_embedding=False (for CrossAttention):")
backbone_7x7 = helpers.models.load_truncated_model("rad", single_embedding=False)
output_7x7 = backbone_7x7(test_input)
print(f"  Output shape: {output_7x7.shape}")
print(f"  Expected: torch.Size([2, 2048, 7, 7])")

assert output_1x1.shape == torch.Size([2, 2048, 1, 1]), "1x1 mode broken!"
assert output_7x7.shape == torch.Size([2, 2048, 7, 7]), "7x7 mode broken!"
print("\n✅ Both modes working correctly!")


Testing single_embedding=True (for Siamese):
  Output shape: torch.Size([2, 2048, 1, 1])
  Expected: torch.Size([2, 2048, 1, 1])

Testing single_embedding=False (for CrossAttention):
  Output shape: torch.Size([2, 2048, 7, 7])
  Expected: torch.Size([2, 2048, 7, 7])

✅ Both modes working correctly!


In [8]:
# Test both modes
print("Testing single_embedding=True (for Siamese):")
backbone_1x1 = helpers.models.load_truncated_model("rad", single_embedding=True)
test_input = torch.randn(2, 3, 224, 224).to(device)
# test_input = torch.randn(2, 1, 224, 224).to(device)
output_1x1 = backbone_1x1(test_input)
print(f"  Output shape: {output_1x1.shape}")
print(f"  Expected: torch.Size([2, 2048, 1, 1])")

print("\nTesting single_embedding=False (for CrossAttention):")
backbone_7x7 = helpers.models.load_truncated_model("rad", single_embedding=False)
output_7x7 = backbone_7x7(test_input)
print(f"  Output shape: {output_7x7.shape}")
print(f"  Expected: torch.Size([2, 2048, 7, 7])")

assert output_1x1.shape == torch.Size([2, 2048, 1, 1]), "1x1 mode broken!"
assert output_7x7.shape == torch.Size([2, 2048, 7, 7]), "7x7 mode broken!"
print("\n✅ Both modes working correctly!")


Testing single_embedding=True (for Siamese):
  Output shape: torch.Size([2, 2048, 1, 1])
  Expected: torch.Size([2, 2048, 1, 1])

Testing single_embedding=False (for CrossAttention):
  Output shape: torch.Size([2, 2048, 7, 7])
  Expected: torch.Size([2, 2048, 7, 7])

✅ Both modes working correctly!


In [17]:
# Test both modes
print("Testing single_embedding=True (for Siamese):")
backbone_1x1 = helpers.models.load_truncated_model("rgb", single_embedding=True)
test_input = torch.randn(2, 3, 224, 224).to(device)
# test_input = torch.randn(2, 1, 224, 224).to(device)
output_1x1 = backbone_1x1(test_input)
print(f"  Output shape: {output_1x1.shape}")
print(f"  Expected: torch.Size([2, 2048, 1, 1])")

print("\nTesting single_embedding=False (for CrossAttention):")
backbone_7x7 = helpers.models.load_truncated_model("rgb", single_embedding=False)
output_7x7 = backbone_7x7(test_input)
print(f"  Output shape: {output_7x7.shape}")
print(f"  Expected: torch.Size([2, 2048, 7, 7])")

assert output_1x1.shape == torch.Size([2, 2048, 1, 1]), "1x1 mode broken!"
assert output_7x7.shape == torch.Size([2, 2048, 7, 7]), "7x7 mode broken!"
print("\n✅ Both modes working correctly!")


Testing single_embedding=True (for Siamese):
  Output shape: torch.Size([2, 2048, 1, 1])
  Expected: torch.Size([2, 2048, 1, 1])

Testing single_embedding=False (for CrossAttention):
  Output shape: torch.Size([2, 2048, 7, 7])
  Expected: torch.Size([2, 2048, 7, 7])

✅ Both modes working correctly!
